# House Price Prediction — Multiple Linear Regression from Scratch

---

**Repository:** Machine Learning from Scratch  
**Notebook:** 01 — House Price Prediction  
**Algorithm:** Multiple Linear Regression with Mini-Batch Gradient Descent  
**No libraries:** NumPy only — zero Scikit-Learn  
**Author:** Ather-ops  

---

## Objective

Implement **Multiple Linear Regression completely from scratch** using only NumPy and Pandas. No `sklearn.linear_model`, no `.fit()`, no black box. Every gradient is computed manually, every parameter update is written explicitly.

This notebook is the foundation of the entire from-scratch curriculum. Every concept built here — forward pass, loss function, gradient computation, parameter update — appears in every model that follows.

---

## What Makes This Different from the Scikit-Learn Repo

| Aspect | Scikit-Learn Repo | This Repo (From Scratch) |
|--------|------------------|-------------------------|
| Model training | `model.fit(X, Y)` — one line | Manual gradient loop across epochs |
| Gradient computation | Hidden inside C code | Written explicitly with NumPy |
| Learning rate | Handled automatically | You set it, you own it |
| Parameter updates | Automatic | `m -= alpha * gradient` written by hand |
| Purpose | Industry pipelines | Deep mathematical understanding |

---

## The Model

We predict house price using three features:

$$\hat{Price} = m_1 \cdot Area + m_2 \cdot Bedrooms + m_3 \cdot Age + b$$

Where $m_1, m_2, m_3$ are weights and $b$ is the bias. All four start at zero and are updated iteratively by gradient descent.

---

## The Loss Function — Mean Squared Error

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

We minimize this by computing the partial derivative of MSE with respect to each parameter and moving each parameter in the direction that reduces loss.

---

## Full Pipeline

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Load dataset |
| 3 | Exploratory Data Analysis |
| 4 | Initialize parameters |
| 5 | Understand gradient descent — the math |
| 6 | Mini-batch gradient descent loop |
| 7 | Track loss and parameters |
| 8 | Final predictions and metrics |
| 9 | Visualization — 6 plots |

---
## Step 1 — Import Libraries

This entire notebook uses only three libraries. The deliberate constraint forces every ML operation to be written explicitly.

| Library | Role | What we do NOT use |
|---------|------|--------------------|
| `pandas` | Load CSV, create DataFrame, store predictions | No Sklearn preprocessing |
| `numpy` | All numerical operations — gradients, matrix math, shuffling | No Sklearn models |
| `matplotlib.pyplot` | All 6 visualizations | No Seaborn shortcuts |

No `sklearn`, no `scipy`, no `statsmodels`. Everything is hand-built.

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

---
## Step 2 — Load Dataset

We load the housing CSV and extract each column as a separate NumPy array. Working with individual arrays (rather than a full matrix) keeps the gradient math explicit and readable — each gradient formula directly references the feature it belongs to.

| Variable | Source column | Role |
|----------|--------------|------|
| `X1` | Area | Feature 1 — house size in sq ft |
| `X2` | Bedrooms | Feature 2 — number of bedrooms |
| `X3` | Age | Feature 3 — age of house in years |
| `Y` | Price | Target — sale price to predict |
| `n` | `len(df)` | Total number of training samples |

In [ ]:
# ==================== Load Data ====================
df = pd.read_csv("housing_sample1.csv")  

# Features and target
X1 = df["Area"].values
X2 = df["Bedrooms"].values
X3 = df["Age"].values
Y  = df["Price"].values

n = len(df)

print("Dataset loaded")
print("="*60)
print(f"Total samples : {n}")
print(f"Features      : Area, Bedrooms, Age")
print(f"Target        : Price")
print("="*60)
print("First 10 rows:")
print(df.head(10).to_string())
print("="*60)

---
## Step 3 — Exploratory Data Analysis

Before touching any parameters, we inspect the raw data to understand distributions, spot outliers, and confirm the feature-target relationships we expect.

**What to look for:**

| Plot | Expected pattern |
|------|------------------|
| Area vs Price | Clear upward trend — the strongest predictor |
| Bedrooms vs Price | Stepwise upward — more rooms, higher price |
| Age vs Price | Downward trend — older house, lower price |

We also print basic statistics to check for missing values and understand value ranges — especially important before setting the learning rate, since large feature magnitudes (Area in thousands) require a very small `alpha`.

In [ ]:
# EDA — statistics and scatter plots
print("Basic Statistics:")
print("="*60)
print(df.describe().round(2))
print("="*60)
print("Missing values:")
print(df.isnull().sum())
print("="*60)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(df["Area"], Y, color="steelblue",
                edgecolors="white", s=60, alpha=0.8)
axes[0].set_xlabel("Area (sq ft)")
axes[0].set_ylabel("Price")
axes[0].set_title("Area vs Price (Before Training)")
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].scatter(df["Bedrooms"], Y, color="darkorange",
                edgecolors="white", s=60, alpha=0.8)
axes[1].set_xlabel("Bedrooms")
axes[1].set_ylabel("Price")
axes[1].set_title("Bedrooms vs Price (Before Training)")
axes[1].grid(True, linestyle="--", alpha=0.5)

axes[2].scatter(df["Age"], Y, color="seagreen",
                edgecolors="white", s=60, alpha=0.8)
axes[2].set_xlabel("Age (years)")
axes[2].set_ylabel("Price")
axes[2].set_title("Age vs Price (Before Training)")
axes[2].grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Exploratory Data Analysis — Raw Feature Relationships",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("="*60)

---
## Step 4 — Initialize Parameters

We initialize all four learnable parameters to zero. This is the standard starting point for linear regression — the model begins with no knowledge and learns entirely from the data through gradient descent.

| Parameter | Initial value | What it controls |
|-----------|--------------|------------------|
| `m1` | 0 | Weight for Area |
| `m2` | 0 | Weight for Bedrooms |
| `m3` | 0 | Weight for Age |
| `b` | 0 | Bias — base price when all features are zero |

**Learning rate `alpha = 0.0000001`:**  
Area values range up to thousands. Bedrooms are single digits. This scale difference means the gradient for Area is much larger in magnitude than the gradient for Bedrooms. A very small learning rate prevents the large Area gradient from causing the parameters to overshoot and diverge. This is exactly why feature scaling exists in the Scikit-Learn repo — but here we handle it through careful alpha selection to keep the math transparent.

**Mini-batch size `batch_size = 10`:**  
Instead of updating parameters once per epoch (batch gradient descent) or once per sample (stochastic gradient descent), we update every 10 samples. This is the industry-standard approach — faster than full-batch, more stable than stochastic.

In [ ]:
# ==================== Initialize Parameters ====================
m1, m2, m3, b = 0, 0, 0, 0
alpha      = 0.0000001  # Learning rate
epochs     = 1000
batch_size = 10

print("Parameters Initialized")
print("="*60)
print(f"m1 (Area weight)     : {m1}")
print(f"m2 (Bedrooms weight) : {m2}")
print(f"m3 (Age weight)      : {m3}")
print(f"b  (Bias)            : {b}")
print("="*60)
print(f"Learning rate (alpha): {alpha}")
print(f"Epochs               : {epochs}")
print(f"Batch size           : {batch_size}")
print(f"Updates per epoch    : {int(np.ceil(n / batch_size))}")
print(f"Total parameter updates: {epochs * int(np.ceil(n / batch_size))}")
print("="*60)

---
## Step 5 — The Gradient Descent Math

This is the core of the entire notebook. Before reading the code, understand the equations.

### Forward Pass — Prediction

$$\hat{y} = m_1 x_1 + m_2 x_2 + m_3 x_3 + b$$

### Loss Function — Mean Squared Error

$$L = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

### Gradients — Partial Derivatives of Loss

We compute how much the loss changes when each parameter changes by a tiny amount:

$$\frac{\partial L}{\partial m_1} = \frac{2}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i) \cdot x_{1i}$$

$$\frac{\partial L}{\partial m_2} = \frac{2}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i) \cdot x_{2i}$$

$$\frac{\partial L}{\partial m_3} = \frac{2}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i) \cdot x_{3i}$$

$$\frac{\partial L}{\partial b} = \frac{2}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)$$

### Parameter Update Rule

$$m_1 \leftarrow m_1 - \alpha \cdot \frac{\partial L}{\partial m_1}$$

$$m_2 \leftarrow m_2 - \alpha \cdot \frac{\partial L}{\partial m_2}$$

$$m_3 \leftarrow m_3 - \alpha \cdot \frac{\partial L}{\partial m_3}$$

$$b \leftarrow b - \alpha \cdot \frac{\partial L}{\partial b}$$

The negative sign is critical — we move **against** the gradient because the gradient points toward increasing loss. We want decreasing loss.

### Mini-Batch vs Batch vs Stochastic

| Method | Update frequency | Pros | Cons |
|--------|-----------------|------|------|
| Batch GD | Once per epoch (full data) | Smooth convergence | Slow on large datasets |
| Stochastic GD | Once per sample | Fast updates | Very noisy |
| Mini-Batch GD | Once per batch (10 here) | Best of both | Hyperparameter: batch size |

---
## Step 6 — Mini-Batch Gradient Descent Loop

The training loop structure:

```
for each epoch:
    shuffle the dataset            <- prevents order bias
    for each mini-batch:
        forward pass               <- compute y_pred
        compute error              <- y_pred - y_actual  
        compute 4 gradients        <- one per parameter
        update 4 parameters        <- subtract alpha * gradient
    compute full-dataset loss      <- track progress
    store loss and parameters      <- for visualization
    print every 100 epochs         <- monitor training
```

**Why shuffle every epoch:**  
If the data is always processed in the same order, the model overfits to that order. Shuffling ensures every mini-batch sees a random mix of houses — making the learned weights generalize better.

**Why track loss on full dataset after each epoch:**  
Mini-batch loss is noisy — it only reflects 10 samples. Full-dataset loss gives a clean picture of whether the model is actually improving across all training examples.

In [ ]:
# ==================== Track Loss and Parameters ====================
loss_history = []
m1_history, m2_history, m3_history, b_history = [], [], [], []

# ==================== Gradient Descent ====================
for epoch in range(epochs):
    # Shuffle dataset
    indices    = np.arange(n)
    np.random.shuffle(indices)
    X1_shuffled = X1[indices]
    X2_shuffled = X2[indices]
    X3_shuffled = X3[indices]
    Y_shuffled  = Y[indices]

    # Mini-batch gradient descent
    for i in range(0, n, batch_size):
        X1_batch = X1_shuffled[i:i+batch_size]
        X2_batch = X2_shuffled[i:i+batch_size]
        X3_batch = X3_shuffled[i:i+batch_size]
        Y_batch  = Y_shuffled[i:i+batch_size]
        batch_n  = len(X1_batch)

        # Predictions
        Y_pred_batch = m1*X1_batch + m2*X2_batch + m3*X3_batch + b

        # Errors
        error_batch = Y_pred_batch - Y_batch

        # Gradients
        gradient_m1 = (2/batch_n) * np.sum(error_batch * X1_batch)
        gradient_m2 = (2/batch_n) * np.sum(error_batch * X2_batch)
        gradient_m3 = (2/batch_n) * np.sum(error_batch * X3_batch)
        gradient_b  = (2/batch_n) * np.sum(error_batch)

        # Update parameters
        m1 -= alpha * gradient_m1
        m2 -= alpha * gradient_m2
        m3 -= alpha * gradient_m3
        b  -= alpha * gradient_b

    # Track total loss on full dataset
    Y_pred_total = m1*X1 + m2*X2 + m3*X3 + b
    loss = np.mean((Y_pred_total - Y)**2)
    loss_history.append(loss)
    m1_history.append(m1)
    m2_history.append(m2)
    m3_history.append(m3)
    b_history.append(b)

    # Print every 100 epochs
    if epoch % 100 == 0:
        print(f"Epoch {epoch:>4}: Loss={loss:.4f}, "
              f"m1={m1:.6f}, m2={m2:.6f}, "
              f"m3={m3:.6f}, b={b:.6f}")

---
## Step 7 — Final Predictions and Metrics

After all 1000 epochs, we compute final predictions on the full training set and evaluate using three metrics.

| Metric | Formula | What it tells us |
|--------|---------|------------------|
| MSE | $\frac{1}{n}\sum(\hat{y} - y)^2$ | Average squared error — same unit as loss during training |
| RMSE | $\sqrt{MSE}$ | Same unit as Price — interpretable as average error in price |
| MAE | $\frac{1}{n}\sum|\hat{y} - y|$ | Average absolute error — less sensitive to large outliers than RMSE |

We also print the final learned parameters. The sign and magnitude of each weight directly reflects what the model learned about each feature's relationship with price:
- `m1` (Area) should be positive and the largest — area drives price most
- `m2` (Bedrooms) should be positive — more rooms, higher price
- `m3` (Age) should be negative — older house, lower price

In [ ]:
# ==================== Final Predictions and Metrics ====================
df["Predicted_Price"] = m1*X1 + m2*X2 + m3*X3 + b
MSE  = np.mean((df["Predicted_Price"] - Y)**2)
RMSE = np.sqrt(MSE)
MAE  = np.mean(np.abs(df["Predicted_Price"] - Y))

print("="*80)
print(f"Training Completed after {epochs} epochs")
print(f"Final Parameters: m1={m1:.6f}, m2={m2:.6f}, m3={m3:.6f}, b={b:.6f}")
print(f"MSE={MSE:.4f}, RMSE={RMSE:.4f}, MAE={MAE:.4f}")
print("="*80)

print("\nFinal learned equation:")
print(f"  Price = {m1:.6f}*Area + {m2:.6f}*Bedrooms + {m3:.6f}*Age + {b:.6f}")
print("="*80)

# Comparison table — first 10 predictions
comparison = pd.DataFrame({
    "Area"           : X1[:10],
    "Bedrooms"       : X2[:10],
    "Age"            : X3[:10],
    "Actual Price"   : Y[:10],
    "Predicted Price": df["Predicted_Price"].values[:10].round(2),
    "Error"          : (Y[:10] - df["Predicted_Price"].values[:10]).round(2)
})
print("\nFirst 10 predictions vs actual:")
print("="*80)
print(comparison.to_string(index=False))
print("="*80)

---
## Step 8 — Visualization

Six plots arranged in a 2x3 grid give a complete picture of training behavior and model performance.

| Panel | Plot | What to look for |
|-------|------|------------------|
| 1 | Loss over epochs | Smooth downward curve — confirms gradient descent is working |
| 2 | Parameters over epochs | All 4 parameters converging to stable values |
| 3 | Actual vs Predicted (Area) | Predicted dots clustering toward actual dots |
| 4 | Actual vs Predicted scatter | Points clustering around the red diagonal line |
| 5 | Residual plot | Random scatter around zero — no systematic pattern |
| 6 | Final prediction line | Regression line passing through the data cloud |

**Panel 1 — Loss curve interpretation:**  
A steep initial drop followed by gradual flattening is ideal. If the curve is flat from epoch 0, the learning rate is too small. If it spikes or diverges, the learning rate is too large.

**Panel 2 — Parameter convergence:**  
Watch for all four lines to stabilize. Unstable oscillating parameters signal a learning rate that needs tuning.

**Panel 5 — Residual plot:**  
A good linear model shows residuals scattered randomly around zero with no pattern. A curved or funnel-shaped residual plot means the relationship is non-linear or variance is non-constant.

In [ ]:
# ==================== Visualization ====================
plt.figure(figsize=(18, 10))

# Panel 1: Loss over epochs
plt.subplot(2, 3, 1)
plt.plot(loss_history, color="blue", linewidth=1.5)
plt.title("Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True)

# Panel 2: Parameters over epochs
plt.subplot(2, 3, 2)
plt.plot(m1_history, label="m1 (Area)")
plt.plot(m2_history, label="m2 (Bedrooms)")
plt.plot(m3_history, label="m3 (Age)")
plt.plot(b_history,  label="b (Bias)")
plt.title("Parameters over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.legend()
plt.grid(True)

# Panel 3: Actual vs Predicted Price (Area axis)
plt.subplot(2, 3, 3)
plt.scatter(df["Area"], Y,                    label="Actual",    color="red",   alpha=0.5)
plt.scatter(df["Area"], df["Predicted_Price"], label="Predicted", color="green", alpha=0.5)
plt.title("Actual vs Predicted (Area)")
plt.xlabel("Area")
plt.ylabel("Price")
plt.legend()
plt.grid(True)

# Panel 4: Actual vs Predicted scatter
plt.subplot(2, 3, 4)
plt.scatter(Y, df["Predicted_Price"], alpha=0.6)
plt.plot([Y.min(), Y.max()], [Y.min(), Y.max()],
         color="red", linestyle="--", label="Perfect Prediction")
plt.title("Actual vs Predicted Price")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.legend()
plt.grid(True)

# Panel 5: Residuals
plt.subplot(2, 3, 5)
residuals = Y - df["Predicted_Price"]
plt.scatter(df["Predicted_Price"], residuals, alpha=0.6)
plt.axhline(y=0, color="red", linestyle="--")
plt.title("Residual Plot")
plt.xlabel("Predicted Price")
plt.ylabel("Residuals")
plt.grid(True)

# Panel 6: Final prediction line
plt.subplot(2, 3, 6)
plt.scatter(df["Area"], Y, label="Actual", color="red", alpha=0.3)
plt.plot(df["Area"], df["Predicted_Price"],
         label="Final Prediction", color="blue", linewidth=2)
plt.title("Final Predictions")
plt.xlabel("Area")
plt.ylabel("Price")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

---
## Step 9 — Loss Convergence Deep Dive

We zoom into two extra views of the loss curve to examine convergence behavior more closely — the early training phase and the final 100 epochs.

In [ ]:
# Loss convergence analysis
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Early phase — first 100 epochs
axes[0].plot(loss_history[:100], color="steelblue", linewidth=2)
axes[0].set_title("Loss — First 100 Epochs (Initial Drop)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].grid(True, linestyle="--", alpha=0.5)

# Final phase — last 100 epochs
axes[1].plot(range(900, 1000), loss_history[900:], color="seagreen", linewidth=2)
axes[1].set_title("Loss — Last 100 Epochs (Convergence Zone)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE Loss")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Loss Curve Convergence Analysis", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Loss Summary")
print("="*60)
print(f"Initial loss  (epoch 0)    : {loss_history[0]:.4f}")
print(f"Loss at epoch 100          : {loss_history[100]:.4f}")
print(f"Loss at epoch 500          : {loss_history[500]:.4f}")
print(f"Final loss    (epoch 999)  : {loss_history[-1]:.4f}")
print(f"Total loss reduction       : {loss_history[0] - loss_history[-1]:.4f}")
print(f"Percentage reduction       : {(1 - loss_history[-1]/loss_history[0])*100:.2f}%")
print("="*60)

---
## Step 10 — Predict Price for a New House

We use the final learned parameters to predict the price of a new house the model has never seen.

**New house:**

| Feature | Value |
|---------|-------|
| Area | 2000 sq ft |
| Bedrooms | 3 |
| Age | 5 years |

No scaling step is needed here — unlike the Scikit-Learn repo, the parameters were learned directly on the raw unscaled feature values. So we apply them directly to raw input.

In [ ]:
# Predict price for a new house using final learned parameters
new_area     = 2000
new_bedrooms = 3
new_age      = 5

new_price = m1 * new_area + m2 * new_bedrooms + m3 * new_age + b

print("New House Prediction")
print("="*60)
print(f"Area           : {new_area} sq ft")
print(f"Bedrooms       : {new_bedrooms}")
print(f"Age            : {new_age} years")
print("="*60)
print(f"Calculation    :")
print(f"  {m1:.6f} x {new_area}")
print(f"+ {m2:.6f} x {new_bedrooms}")
print(f"+ {m3:.6f} x {new_age}")
print(f"+ {b:.6f}")
print("="*60)
print(f"Predicted Price: {new_price:.2f}")
print("="*60)

---
## Pipeline Summary

| Step | Action | Key concept |
|------|--------|-------------|
| 1 | Import pandas, numpy, matplotlib | No Scikit-Learn anywhere |
| 2 | Load CSV, extract feature arrays | Raw numpy arrays — no preprocessing |
| 3 | EDA — scatter plots and statistics | Same as any production pipeline |
| 4 | Initialize m1, m2, m3, b = 0 | Parameters start with zero knowledge |
| 5 | Understand gradient math | Forward pass, MSE loss, 4 partial derivatives |
| 6 | Mini-batch GD loop — 1000 epochs | Shuffle + batch + compute gradients + update |
| 7 | Final predictions and MSE/RMSE/MAE | Same metrics as Scikit-Learn notebook |
| 8 | 6 visualizations | Loss curve, parameter convergence, residuals |
| 9 | Loss convergence deep dive | Early drop vs final stability |
| 10 | New house prediction | Direct formula application — no scaler needed |

---

## What This Notebook Proves

The Scikit-Learn notebook (in the companion repo) produces the same predictions with `model.fit(X, Y)` in one line. This notebook shows exactly what that one line is doing internally:

- 1000 passes through the data
- Thousands of gradient computations
- Thousands of parameter updates
- A loss that started high and was systematically driven down

Understanding this is what separates an engineer who can use ML tools from one who can debug them, improve them, and build new ones.

---

**Next notebook:** `02_linear_regression_scratch.ipynb` — implemented from scratch